# Reranker GPU 속도 테스트 + N 스윕

로컬(CPU) 환경에서 `BAAI/bge-reranker-v2-m3`로 문항당 후보 20개를 재정렬하는 데 **평균 40초**가 걸려서, 120문항 전체로 N(재정렬 후보 개수: 10/15/20/25/30)을 스윕하는 실험을 CPU로는 하기 어려웠습니다. GPU에서는 이게 실제로 얼마나 빨라지는지 확인하고, 빨라진다면 **진짜 목적인 N 스윕(어디까지 reranker를 허용할지)을 이 노트북에서 바로 끝내는 것**이 목표입니다.

**실행 전에 GPU 런타임으로 바꿔주세요**: 상단 메뉴 `런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU`

구성:
1. **빠른 합성 벤치마크** — 프로젝트 데이터 없이 바로 GPU 속도만 확인 (지금 바로 실행 가능)
2. **실제 프로젝트로 N 스윕** — RAG_project3 폴더를 업로드해서 `evaluation/tune_reranker.py`(120문항, N=10/15/20/25/30)를 실제로 돌려 최적 N을 확정

## 0. GPU 확인

In [1]:
!nvidia-smi

import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU가 없습니다 — 런타임 유형을 GPU로 바꿔서 다시 실행하세요.")

Tue Aug  4 00:57:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. 의존성 설치

`FlagEmbedding`을 그냥 설치하면 `transformers`를 너무 새 버전(5.x)으로 올려서 `XLMRobertaTokenizer has no attribute 'prepare_for_model'` 에러가 납니다. 호환되는 버전으로 고정해서 설치합니다.

**이미 한 번 에러를 본 경우**: 아래 셀만 다시 실행해도 안 고쳐질 수 있어요(이미 잘못된 버전이 메모리에 로드돼 있어서). `런타임 → 세션 다시 시작` 한 뒤 위에서부터 순서대로 다시 실행하세요.

In [2]:
!pip install -q FlagEmbedding "transformers==4.46.3" "tokenizers==0.20.3"

import os
os.environ["USE_TF"] = "0"  # transformers가 불필요하게 tensorflow를 임포트하려다 나는 충돌 방지

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 160.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Part 1 — 빠른 합성 벤치마크

로컬 CPU 테스트 때와 똑같은 조건(문항 1개당 후보 20개, 비슷한 길이의 텍스트)으로 맞춰서, **GPU vs CPU 배수**를 바로 비교할 수 있게 했습니다. 여기서 확실히 빨라지는 게 보이면 바로 Part 2(진짜 N 스윕)로 넘어가세요.

In [3]:
import time
from FlagEmbedding import FlagReranker

reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)

# 로컬 CPU 테스트와 동일한 조건: 후보 20개, 실제 청크와 비슷한 길이(제목+섹션+본문 300자)
question = "착오송금 반환지원은 어디서 신청하나요?"
dummy_chunk_text = (
    "착오송금 반환지원 FAQ\n온라인으로만 신청이 가능한가요?\n"
    + "신청 방법 및 준비물 관련 상세 안내 내용입니다. " * 15
)[:300]
pairs = [(question, dummy_chunk_text) for _ in range(20)]

# 워밍업 1회(모델 로드 직후 첫 호출은 느릴 수 있어서 측정에서 제외)
_ = reranker.compute_score(pairs[:2], normalize=False, max_length=256)

start = time.perf_counter()
scores = reranker.compute_score(pairs, normalize=False, max_length=256)
elapsed = time.perf_counter() - start

print(f"후보 {len(pairs)}개 채점: {elapsed:.2f}초 ({elapsed/len(pairs)*1000:.1f}ms/쌍)")
print(f"문항 1개(후보 20개 기준) 예상 소요: {elapsed:.2f}초")
print(f"로컬 CPU 대비 배수: {40/elapsed:.1f}배 빠름" if elapsed > 0 else "")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


후보 20개 채점: 0.09초 (4.6ms/쌍)
문항 1개(후보 20개 기준) 예상 소요: 0.09초
로컬 CPU 대비 배수: 437.3배 빠름


## 3. Part 2 — 실제 프로젝트로 N 스윕 (진짜 목적)

`evaluation/tune_reranker.py`를 그대로 돌립니다. 이 스크립트는:
- 120문항 전체에 대해 Hybrid(pool=30) 융합 결과 top-30을 가져오고
- 후보 30개를 reranker로 한 번씩만 채점(캐싱)한 뒤
- N ∈ {10, 15, 20, 25, 30}으로 슬라이스만 바꿔가며 스윕해서
- mrr@10 기준 최적 N을 찾아줍니다 (rerank 없는 baseline과도 비교)

**준비물**: 로컬 `RAG_project3` 폴더 전체(코드 + `data/` 안의 jsonl 3종·평가 xlsx)를 zip으로 압축해두세요.

In [4]:
from google.colab import files
import zipfile, os
from pathlib import Path

print("RAG_project3 폴더를 압축한 zip 파일을 업로드하세요...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

extract_to = Path("/content/rag_project3_extracted")
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_to)

project_root = None
for dirpath, dirnames, _ in os.walk(extract_to):
    if "core" in dirnames and "evaluation" in dirnames:
        project_root = Path(dirpath)
        break

if project_root is None:
    print("못 찾음 — 아래 트리 구조를 확인하세요:")
    for root, dirs, _ in os.walk(extract_to):
        depth = str(root).replace(str(extract_to), "").count(os.sep)
        if depth <= 3:
            print("  " * depth + os.path.basename(root) + "/")
    raise RuntimeError("core/, evaluation/ 폴더를 못 찾았어요. 위 트리를 보고 알려주세요.")

os.chdir(project_root)
print("프로젝트 루트로 이동 완료:", project_root)

RAG_project3 폴더를 압축한 zip 파일을 업로드하세요...


Saving RAG_project3.zip to RAG_project3.zip
프로젝트 루트로 이동 완료: /content/rag_project3_extracted/RAG_project3


In [5]:
%cd /content/RAG_project3/RAG_project3/
# requirements.txt를 통째로 설치하면 Colab 자체가 쓰는 ipython/pandas/numpy/huggingface-hub
# 버전이랑 충돌합니다. numpy/pandas/ipython은 Colab에 이미 충분히 최신이라
# 진짜 새로 필요한 openai만 설치합니다. (FlagEmbedding·transformers·tokenizers는
# Part 1의 1번 셀에서 이미 버전 고정해서 설치해뒀습니다.)
!pip install -q "openai>=1.68,<2"

[Errno 2] No such file or directory: '/content/RAG_project3/RAG_project3/'
/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 53.6 MB/s eta 0:00:00


In [6]:
import getpass, os
os.environ["HCX_API_KEY"] = getpass.getpass("HCX_API_KEY 입력: ")

HCX_API_KEY 입력: ··········


In [8]:
# 120문항 전체 × N={10,15,20,25,30} 스윕을 실제로 실행합니다.
# 진행률(문항별 경과·평균·예상잔여시간)이 실시간으로 출력됩니다.
!pip install -q kiwipiepy rank_bm25
!python -u evaluation/tune_reranker.py

설정 완료
- Chat 모델: HCX-005
- 질문 임베딩 모델: bge-m3
- Top-K: 5
- 최소 유사도: 0.3
- 업무 필터: None
로컬 데이터 사용: /content/rag_project3_extracted/RAG_project3/data
폴더 복사 완료: /content/kdic_rag_baseline/uploaded_output
로드 완료
- documents: 87 /content/kdic_rag_baseline/uploaded_output/documents.jsonl
- chunks: 427 /content/kdic_rag_baseline/uploaded_output/chunks.jsonl
- embeddings: 427 /content/kdic_rag_baseline/uploaded_output/chunk_embeddings_hcx.jsonl
무결성 검사 통과
- 청크-임베딩 연결: 427 건
- 임베딩 차원: {1024}
- 저장 임베딩 모델: {'bge-m3'}
- 업무 목록:
  - 고객 미수령금 신청
  - 예금보험금 안내
  - 예금자보호제도
  - 은닉재산 신고
  - 착오송금 반환 신청
  - 채무조정 안내
BM25 인덱스 준비 완료: 427 청크
HCX 클라이언트 설정 완료
- Base URL: https://clovastudio.stream.ntruss.com/v1/openai
- Chat 모델: HCX-005
- 질문 임베딩 모델: bge-m3
- API 키: 등록됨
HCX 호출 함수 준비 완료
Baseline 검색·답변 함수 준비 완료
Hybrid 검색(Weighted-sum, Min-Max) 준비 완료
Reranker(BGE-Reranker-v2-m3) 모듈 준비 완료
평가 하니스 준비 완료
평가 문항 수(전체): 120
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call

## 4. 결론 판단 기준

위 셀 마지막에 출력되는 표(`N 스윕 결과`)와 `>>> 최고 조합: ...` 줄을 그대로 로컬 세션에 알려주세요.

- **baseline(rerank 없음) 대비 어느 N에서 mrr@10/hit@3가 제일 좋은지**
- **문항 1건당 걸린 평균 시간**(진행 로그의 `평균 X.X초/건`)이 실시간 채팅에 쓸 만한 수준인지(2~3초 이내면 여유, 아니면 여전히 조건부/옵션 유지)

이 두 가지로 `retrieval/reranker.py`의 최종 N값과 `USE_RERANKER` 플래그를 확정합니다.

In [9]:
from google.colab import files
import shutil, os

uploaded = files.upload()  # 로컬 RAG_project3/evaluation/tune_reranker.py 선택
shutil.move(list(uploaded.keys())[0], "evaluation/tune_reranker.py")
print("교체 완료")

os.system("python -u evaluation/tune_reranker.py")

Saving tune_reranker.py to tune_reranker.py
교체 완료


0